# jtagent — QLoRA (Qwen2.5-7B) · consolidated · 2026-09-26

Run on an **A100**. Standalone notebook: mount Drive, then run top-to-bottom. TAN folder `1ondyw5YrwXpE6jV48nYpRlg4Z1QkZWUB`.

Replaces the old one-shot E2E + experimental cells with one clean reduced-set, dynamic-padding flow.

In [ ]:
# Mount Drive first (with stale-mount cleanup - repairs a broken /content/drive).
import shutil
from pathlib import Path
from google.colab import drive
mp = Path("/content/drive"); mydrive = mp / "MyDrive"
if mp.exists() and not mydrive.exists():
    stale = Path("/tmp/drive_stale_local")
    if stale.exists(): shutil.rmtree(stale, ignore_errors=True)
    shutil.move(str(mp), str(stale)); shutil.rmtree(stale, ignore_errors=True)
    print("removed_stale_local_drive", True)
drive.mount("/content/drive")
print("MyDrive", mydrive.exists())

## jtagent QLoRA — Qwen2.5-7B reduced-set (dynamic padding) · updated 2026-09-26

One consolidated pipeline (replaces the old one-shot E2E + experimental cells).

**Flow:** setup → data prep (Hub → per-category jsonl) → subsample to **5,001 rows** → detached QLoRA train + **Q4_K_M** GGUF export → watch → publish **private** → device packs → optional Claude Agent SDK agent.

**Key fix — PR #17 (merged):** `qlora_train.py` tokenizes with `padding=False` + `DataCollatorForLanguageModeling(mlm=False)` (per-batch padding, pad masked to −100) instead of padding every row to 2048 and labelling the pad tokens. It also adds env-overridable `max_steps` / `save_steps` / `save_total_limit` + resume. It's on `main`, so the plain clone in the setup cell picks it up.

**Subsample (the composition that produced the published adapter):** keep every non-bulk category in full — `browser_data` (~1,049) + the self-knowledge Q&A (whatsapp / sandbox_ops / devices / …) — and cap the bulk `extracted_text` (file-change metadata) so the **deduped** grand total is exactly **5,001** (→ ~3,945 extracted). A full backup is taken before subsampling.

Verified results are in the final cell.

In [ ]:
!nvidia-smi -L
!git clone --depth 1 https://github.com/jaytipargal/jayti /content/jayti
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF >/dev/null && \
 cmake --build /content/llama.cpp/build -j"$(nproc)" --target llama-quantize >/dev/null && echo "llama-quantize built"
!pip -q install "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" "bitsandbytes>=0.43" gguf sentencepiece protobuf

import os
os.environ.update({
    "JTAGENT_ROOT": "/content/TAN/jtagent",
    "JTAGENT_BASE_MODEL": "Qwen/Qwen2.5-7B-Instruct",
    "JTAGENT_QUANT": "Q4_K_M",
    "LLAMA_CPP_DIR": "/content/llama.cpp",
    "JTAGENT_MAX_STEPS": "1500",         # bounded, resumable run
    "JTAGENT_SAVE_STEPS": "200",
    "JTAGENT_SAVE_TOTAL_LIMIT": "2",
})
from huggingface_hub import login; login()   # HF write token — needed for the PRIVATE publish later

In [ ]:
# Data prep: pull the ingestion dataset from the Hub -> redacted per-category jsonl.
import sys, glob, json
from pathlib import Path
sys.path[:0] = ["/content/jayti/sandbox/jtagent", "/content/jayti/scripts"]
import segment_jsonl
root = Path("/content/TAN/jtagent"); (root / "training").mkdir(parents=True, exist_ok=True)

from huggingface_hub import snapshot_download
snap = snapshot_download("jtagent/jt-agent-data", repo_type="dataset",
                         allow_patterns=["training/daily_ingestion/**"])
items = []
for f in glob.glob(f"{snap}/training/daily_ingestion/**/*.jsonl", recursive=True):
    for line in open(f, encoding="utf-8"):
        s = line.strip()
        if s:
            try: items.append(json.loads(s))
            except json.JSONDecodeError: pass
chunks = [c for c in (segment_jsonl._item_to_chunk(it) for it in items) if c]
chunks = segment_jsonl.redact_chunks(chunks)          # redacts phone numbers / secrets
m = segment_jsonl.write_category_jsonl(root, chunks)
print("raw items:", len(items), "| training rows:", m["total_chunks"], "| categories:", list(m["categories"]))

In [ ]:
# Reduced set = exactly 5,001 rows: keep every non-bulk category in full; cap the
# bulk file-change category (training/extracted_text) so the DEDUPED total == 5001.
import json, random, shutil, time
from pathlib import Path
import qlora_train                                    # for the exact dedup identity
TR, TARGET, SEED, PAD = root / "training", 5001, 42, "pad"

def key(r):
    instr = str(r.get("input") or r.get("instruction") or "").strip()
    return (instr, qlora_train._coerce_output(r.get("output", ""))) if instr else None

bk = root / f"training_backup_full_{time.strftime('%Y%m%d_%H%M%S')}"
shutil.copytree(TR, bk); print("full backup:", bk)

bulk = next(iter((TR / "extracted_text").glob("*.jsonl")))
bak = bulk.with_suffix(".full.jsonl.bak")
if not bak.exists(): shutil.copy2(bulk, bak)

seen = set()                                          # deduped identities of kept-in-full categories
for d in sorted(p for p in TR.iterdir() if p.is_dir()):
    if d.name == "extracted_text": continue
    for f in d.glob("*.jsonl"):
        if f.name.endswith(".bak"): continue
        for s in f.read_text(encoding="utf-8", errors="replace").splitlines():
            try: r = json.loads(s.strip())
            except Exception: continue
            if r.get("category") == PAD: continue
            k = key(r)
            if k: seen.add(k)
cap = TARGET - len(seen)
print("kept-in-full (deduped):", len(seen), "-> extracted_text cap:", cap)

pads, real = [], []
for s in bak.read_text(encoding="utf-8", errors="replace").splitlines():
    try: r = json.loads(s.strip())
    except Exception: continue
    (pads if r.get("category") == PAD else real).append(s.strip())
random.seed(SEED); random.shuffle(real)
kept, seen_ex = [], set(seen)
for s in real:
    if len(kept) >= cap: break
    try: k = key(json.loads(s))
    except Exception: continue
    if k and k not in seen_ex: seen_ex.add(k); kept.append(s)
bulk.write_text("\n".join(pads + kept) + "\n", encoding="utf-8")
n = len(list(qlora_train.iter_training_rows(TR)))
print(f"trainer will see {n} rows (target {TARGET}); extracted_text = {len(kept)} sampled")

In [ ]:
%%writefile /content/TAN/jtagent/run_reduced.py
# Detached driver: train (dynamic padding, max_steps=1500) -> remove incompatible
# torchao (0.10.0 breaks peft's LoRA merge) -> Q4_K_M GGUF export. Survives a
# browser disconnect (the 84k-row run died that way). Watch via the next cell.
import json, subprocess, sys, traceback
sys.path.insert(0, "/content/jayti/sandbox/jtagent")
ROOT = "/content/TAN/jtagent"; DONE = ROOT + "/EXPORT_DONE.json"
try:
    import qlora_train, export_gguf
    cfg = qlora_train.build_config({"base_model": "Qwen/Qwen2.5-7B-Instruct", "max_steps": 1500})
    print("CONFIG", json.dumps(cfg), flush=True)
    res = qlora_train.train_qlora(ROOT, config=cfg, ensure_deps=False)
    print("TRAIN", res.get("status"), res.get("adapter"), flush=True)
    assert res.get("status") == "completed", res
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
    rep = export_gguf.export(res["adapter"], "Qwen/Qwen2.5-7B-Instruct",
                             ROOT + "/gguf", quant="Q4_K_M", llama_cpp_dir="/content/llama.cpp")
    json.dump({"status": "completed", "adapter": res["adapter"], **rep}, open(DONE, "w"), indent=2)
    print("ADAPTER_PATH:", res["adapter"], flush=True)
    print("GGUF_PATH:", rep.get("quant_gguf"), flush=True)
    print("DRIVER_DONE", flush=True)
except Exception as e:
    json.dump({"status": "error:" + type(e).__name__, "error": str(e),
               "trace": traceback.format_exc()}, open(DONE, "w"), indent=2)
    print("DRIVER_ERROR:", e, flush=True); raise

In [ ]:
# Launch the driver detached (nohup) so it outlives this cell / a disconnect.
import subprocess
from pathlib import Path
ROOT = Path("/content/TAN/jtagent"); LOG = ROOT / "agent_train_export.log"
open(LOG, "w").close()
subprocess.Popen(["bash", "-lc", f"nohup python3 -u {ROOT}/run_reduced.py > {LOG} 2>&1 &"])
print("launched detached driver -> run the WATCH cell to stream", LOG)

In [ ]:
# Watch the detached run (reads the log only; safe). Stops when the driver exits.
import re, time, glob
from pathlib import Path
LOG = Path("/content/TAN/jtagent/agent_train_export.log")
def alive():
    for p in glob.glob("/proc/[0-9]*/cmdline"):
        try:
            if "run_reduced.py" in open(p).read(): return True
        except OSError: pass
    return False
last = ""
while True:
    t = LOG.read_text(errors="replace").replace("\r", "\n") if LOG.exists() else ""
    step = re.findall(r"\d+/\d+ \[[^\]]*\]", t)
    loss = re.findall(r"'loss': [0-9.]+", t)
    line = f"{step[-1] if step else '-'} | {loss[-1] if loss else '-'}"
    if line != last: print(time.strftime("%H:%M:%S"), line, flush=True); last = line
    if not alive(): break
    time.sleep(10)
for ln in (LOG.read_text(errors="replace").splitlines() if LOG.exists() else []):
    if any(m in ln for m in ("ADAPTER_PATH:", "GGUF_PATH:", "DRIVER_DONE", "DRIVER_ERROR")):
        print(ln)

In [ ]:
# Publish adapter + Q4_K_M GGUF to a PRIVATE HF repo (go4garage01/jt-agent-lora,
# versioned path adapter/<run>/). hf_publish is private-by-default — never pass --public.
import glob, subprocess
adapter = sorted(glob.glob("/content/TAN/jtagent/adapters/adapter_*_qlora/final"))[-1]
gguf    = sorted(glob.glob("/content/TAN/jtagent/gguf/*.q4_k_m.gguf"))[-1]
subprocess.run(["python3", "/content/jayti/sandbox/jtagent/hf_publish.py",
                "--adapter", adapter, "--gguf", gguf], check=True)

In [ ]:
# Device packs -> point S24 / VivoBook / PC at the new adapter + GGUF.
import sys, glob; sys.path.insert(0, "/content/jayti/sandbox/jtagent")
from pack_devices import pack_devices
adapter = sorted(glob.glob("/content/TAN/jtagent/adapters/adapter_*_qlora/final"))[-1]
gguf    = sorted(glob.glob("/content/TAN/jtagent/gguf/*.q4_k_m.gguf"))[-1]
print(pack_devices("/content/TAN/jtagent",
      extra={"adapter": adapter, "gguf": gguf, "base_model": "Qwen/Qwen2.5-7B-Instruct"}))

In [ ]:
# Optional: run jtagent as a Claude Agent SDK agent (Claude-backed, locked to one
# retrieval tool over the redacted corpus). Needs an Anthropic key (sk-ant-...) in a
# Colab secret named ANTHROPIC_API_KEY - NOT the EKA_* device.env hub keys.
!pip -q install claude-agent-sdk anyio && npm i -g @anthropic-ai/claude-code >/dev/null
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
!python /content/jayti/sandbox/jtagent/jtagent_sdk_agent.py "What devices are registered for jtagent, and what does each run?"

## ✅ Verified results — run of 2026-09-25

| | |
|---|---|
| **Adapter** | `TAN/jtagent/adapters/adapter_2026-09-25_qlora/final/` (161.5 MB) |
| **Provenance** | Qwen2.5-7B-Instruct · QLoRA r16/α32 · 5,001 rows · 1,500 steps (~4.8 ep) · train_loss 0.96 |
| **GGUF (Q4_K_M)** | `TAN/jtagent/gguf/adapter_2026-09-25_qlora.q4_k_m.gguf` · 4,683,073,664 B · sha256 `4802957f…86be75` · loads+generates in llama.cpp |
| **Published (private)** | `go4garage01/jt-agent-lora` → `adapter/adapter_2026-09-25_qlora/` + `gguf/` · verified `private=True` |
| **Code fix** | PR #17 (merged) — label masking + step/save/resume |
| **SDK agent** | `sandbox/jtagent/jtagent_sdk_agent.py` — Claude Agent SDK, `claude-sonnet-5`, locked to one retrieval tool over the redacted corpus |
| **f16 GGUF** | local-only, regenerable intermediate — not published |

**Deploy to VivoBook (Ollama):** authorize rclone once — headless `edge_deploy/configure_rclone_sa.ps1 <sa-key.json>` (or interactive `rclone config reconnect jtagent_tan:`) → `edge_deploy/vivobook/pull_and_build.ps1` → `ollama run jtagent`. **S24:** `edge_deploy/s24/pull.sh` + llama.cpp; a 7B Q4 needs ~6 GB free RAM. See `edge_deploy/README.md`.